# 🏗️ YT8M Genre Classifier — Stage 4: Model Architecture
Двухветочная модель с cross-modal attention fusion. Visual(1024) + Audio(128) → 12 классов.

In [1]:
# ============================================================
# STAGE 4.0 — Импорты
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import json
from pathlib import Path

STAGE3_DIR = Path(r'C:\src\ai\video_classifier\data2\stage3')
STAGE4_DIR = Path(r'C:\src\ai\video_classifier\data2\stage4')
STAGE4_DIR.mkdir(parents=True, exist_ok=True)

with open(STAGE3_DIR / 'config.json') as f:
    cfg = json.load(f)

DIM_VISUAL  = cfg['dim_visual']   # 1024
DIM_AUDIO   = cfg['dim_audio']    # 128
N_CLASSES   = cfg['n_classes']    # 12
GENRES      = cfg['genres']
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'✅ Config loaded:')
print(f'   DIM_VISUAL = {DIM_VISUAL}')
print(f'   DIM_AUDIO  = {DIM_AUDIO}')
print(f'   N_CLASSES  = {N_CLASSES}')
print(f'   DEVICE     = {DEVICE}')


✅ Config loaded:
   DIM_VISUAL = 1024
   DIM_AUDIO  = 128
   N_CLASSES  = 12
   DEVICE     = cpu


In [2]:
# ============================================================
# STAGE 4.1 — Строительные блоки
# ============================================================

class ResidualBlock(nn.Module):
    """Residual block: Linear -> BN -> GELU -> Dropout -> Linear + skip."""
    def __init__(self, dim, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim),
        )
        self.act = nn.GELU()

    def forward(self, x):
        return self.act(x + self.net(x))


class VisualBranch(nn.Module):
    """1024 -> 256 -> 2x ResBlock -> 256.
    Два резблока т.к. визуальные фичи богатые (PCA=907 для 95%).
    """
    def __init__(self, in_dim=1024, embed_dim=256, dropout=0.3):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(in_dim, embed_dim),
            nn.BatchNorm1d(embed_dim),
            nn.GELU(),
        )
        self.res1 = ResidualBlock(embed_dim, dropout)
        self.res2 = ResidualBlock(embed_dim, dropout)

    def forward(self, x):
        return self.res2(self.res1(self.proj(x)))


class AudioBranch(nn.Module):
    """128 -> 128 -> 1x ResBlock -> 128."""
    def __init__(self, in_dim=128, embed_dim=128, dropout=0.3):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(in_dim, embed_dim),
            nn.BatchNorm1d(embed_dim),
            nn.GELU(),
        )
        self.res = ResidualBlock(embed_dim, dropout)

    def forward(self, x):
        return self.res(self.proj(x))


class CrossModalAttention(nn.Module):
    """Два токена [visual, audio] -> MultiheadAttention -> concat -> 256."""
    def __init__(self, visual_dim=256, audio_dim=128,
                 fusion_dim=256, n_heads=4, dropout=0.2):
        super().__init__()
        # Приводим оба к fusion_dim
        self.proj_v = nn.Linear(visual_dim, fusion_dim)
        self.proj_a = nn.Linear(audio_dim,  fusion_dim)
        self.attn   = nn.MultiheadAttention(
            embed_dim=fusion_dim, num_heads=n_heads,
            dropout=dropout, batch_first=True
        )
        self.norm   = nn.LayerNorm(fusion_dim)
        self.out_dim = fusion_dim * 2  # concat visual + audio attended

    def forward(self, v, a):
        # v: (B, visual_dim)  a: (B, audio_dim)
        v_tok = self.proj_v(v).unsqueeze(1)   # (B, 1, fusion_dim)
        a_tok = self.proj_a(a).unsqueeze(1)   # (B, 1, fusion_dim)
        tokens = torch.cat([v_tok, a_tok], dim=1)  # (B, 2, fusion_dim)
        out, attn_w = self.attn(tokens, tokens, tokens)
        out = self.norm(out + tokens)          # residual
        # concat оба токена
        fused = out.reshape(out.size(0), -1)   # (B, fusion_dim*2)
        return fused, attn_w


print('✅ Блоки определены: ResidualBlock, VisualBranch, AudioBranch, CrossModalAttention')


✅ Блоки определены: ResidualBlock, VisualBranch, AudioBranch, CrossModalAttention


In [3]:
# ============================================================
# STAGE 4.2 — Полная модель
# ============================================================

class YT8MClassifier(nn.Module):
    """
    Visual(1024) ──► VisualBranch(256) ──┐
                                          ├─► CrossModalAttention(512) ──► MLP ──► 12
    Audio(128)   ──► AudioBranch(128)  ──┘
    """
    def __init__(self,
                 dim_visual=1024,
                 dim_audio=128,
                 n_classes=12,
                 visual_embed=256,
                 audio_embed=128,
                 fusion_dim=256,
                 n_heads=4,
                 dropout=0.35):
        super().__init__()
        self.visual_branch = VisualBranch(dim_visual,  visual_embed, dropout)
        self.audio_branch  = AudioBranch(dim_audio,   audio_embed,  dropout)
        self.fusion        = CrossModalAttention(
            visual_dim=visual_embed, audio_dim=audio_embed,
            fusion_dim=fusion_dim,   n_heads=n_heads, dropout=dropout*0.5
        )
        fused_dim = self.fusion.out_dim   # 512

        self.classifier = nn.Sequential(
            nn.Linear(fused_dim, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(128, n_classes),
        )

    def forward(self, x_visual, x_audio):
        v = self.visual_branch(x_visual)
        a = self.audio_branch(x_audio)
        fused, attn_w = self.fusion(v, a)
        logits = self.classifier(fused)
        return logits, attn_w

    def predict(self, x_visual, x_audio):
        """Возвращает probs и attention weights."""
        with torch.no_grad():
            logits, attn_w = self.forward(x_visual, x_audio)
            return F.softmax(logits, dim=-1), attn_w


# Инициализируем
model = YT8MClassifier(
    dim_visual=DIM_VISUAL,
    dim_audio=DIM_AUDIO,
    n_classes=N_CLASSES,
).to(DEVICE)

# Считаем параметры
total_params = sum(p.numel() for p in model.parameters())
train_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
size_mb      = total_params * 4 / 1024**2

print(f'🏗️  YT8MClassifier:')
print(f'   Всего параметров   : {total_params:,}  ({size_mb:.2f} MB)')
print(f'   Trainable          : {train_params:,}')
print(f'   Train samples      : {cfg["n_train"]:,}')
print(f'   Param/sample ratio : {train_params / cfg["n_train"]:.1f}')
print()

# Побранчевая статистика
def count_params(module):
    return sum(p.numel() for p in module.parameters())

print(f'   VisualBranch   : {count_params(model.visual_branch):>10,}')
print(f'   AudioBranch    : {count_params(model.audio_branch):>10,}')
print(f'   CrossAttn      : {count_params(model.fusion):>10,}')
print(f'   Classifier MLP : {count_params(model.classifier):>10,}')


🏗️  YT8MClassifier:
   Всего параметров   : 1,107,468  (4.22 MB)
   Trainable          : 1,107,468
   Train samples      : 24,945
   Param/sample ratio : 44.4

   VisualBranch   :    528,128
   AudioBranch    :     50,304
   CrossAttn      :    362,496
   Classifier MLP :    166,540


In [4]:
# ============================================================
# STAGE 4.3 — Forward pass проверка
# ============================================================
model.eval()
dummy_v = torch.randn(8, DIM_VISUAL).to(DEVICE)
dummy_a = torch.randn(8, DIM_AUDIO).to(DEVICE)

with torch.no_grad():
    logits, attn_w = model(dummy_v, dummy_a)

print(f'✅ Forward pass OK:')
print(f'   Input  visual : {dummy_v.shape}')
print(f'   Input  audio  : {dummy_a.shape}')
print(f'   Output logits : {logits.shape}   (batch=8, classes=12)')
print(f'   Attn weights  : {attn_w.shape}   (batch=8, heads=4, 2, 2)')
print(f'   Logits range  : [{logits.min().item():.3f}, {logits.max().item():.3f}]')

# Проверяем softmax суммируется в 1
probs = torch.softmax(logits, dim=-1)
print(f'   Probs sum     : {probs.sum(dim=-1).mean().item():.6f}  (должно быть ~1.0)')


✅ Forward pass OK:
   Input  visual : torch.Size([8, 1024])
   Input  audio  : torch.Size([8, 128])
   Output logits : torch.Size([8, 12])   (batch=8, classes=12)
   Attn weights  : torch.Size([8, 2, 2])   (batch=8, heads=4, 2, 2)
   Logits range  : [-0.134, 0.192]
   Probs sum     : 1.000000  (должно быть ~1.0)


In [5]:
# ============================================================
# STAGE 4.4 — Loss, Optimizer, Scheduler
# ============================================================
import torch
import numpy as np

# FocalLoss — лучше справляется с остаточным дисбалансом
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha     = alpha    # tensor shape (n_classes,) или None
        self.gamma     = gamma
        self.reduction = reduction

    def forward(self, logits, targets):
        ce    = F.cross_entropy(logits, targets, weight=self.alpha, reduction='none')
        pt    = torch.exp(-ce)
        focal = (1 - pt) ** self.gamma * ce
        return focal.mean() if self.reduction == 'mean' else focal


# Загружаем class_weights
class_weights = torch.load(STAGE3_DIR / 'class_weights.pt', weights_only=True).to(DEVICE)

criterion = FocalLoss(alpha=class_weights, gamma=2.0)

# Optimizer
N_EPOCHS    = 100
LR          = 3e-4
WEIGHT_DECAY = 1e-3
PATIENCE    = 15      # early stopping
N_TRAIN_STEPS = cfg['n_train'] // cfg['batch_size']  # steps per epoch

optimizer = torch.optim.AdamW(model.parameters(),
                               lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=LR,
    epochs=N_EPOCHS,
    steps_per_epoch=N_TRAIN_STEPS,
    pct_start=0.1,          # 10% warmup
    anneal_strategy='cos',
)

print(f'✅ Training setup:')
print(f'   Loss      : FocalLoss(gamma=2.0, weighted)')
print(f'   Optimizer : AdamW  lr={LR}  wd={WEIGHT_DECAY}')
print(f'   Scheduler : OneCycleLR  epochs={N_EPOCHS}  warmup=10%')
print(f'   Patience  : {PATIENCE} epochs (val macro-F1)')
print(f'   Steps/epoch: {N_TRAIN_STEPS}')

# Сохраняем
torch.save(model.state_dict(), STAGE4_DIR / 'model_init.pt')

train_config = {
    'n_epochs'    : N_EPOCHS,
    'lr'          : LR,
    'weight_decay': WEIGHT_DECAY,
    'patience'    : PATIENCE,
    'batch_size'  : cfg['batch_size'],
    'optimizer'   : 'AdamW',
    'scheduler'   : 'OneCycleLR',
    'loss'        : 'FocalLoss(gamma=2.0)',
    'total_params': total_params,
    'param_sample_ratio': round(train_params / cfg['n_train'], 1),
}
with open(STAGE4_DIR / 'train_config.json', 'w') as f:
    json.dump(train_config, f, indent=2)

print()
print('=' * 55)
print('       STAGE 4 — COMPLETE')
print('=' * 55)
print(f'  Модель     : YT8MClassifier')
print(f'  Параметры  : {total_params:,}  ({size_mb:.2f} MB)')
print(f'  Param/sample: {train_params/cfg["n_train"]:.1f}x')
print(f'  Loss       : FocalLoss gamma=2.0')
print(f'  Optimizer  : AdamW lr=3e-4')
print(f'  Scheduler  : OneCycleLR {N_EPOCHS} epochs')
print('=' * 55)
print('✅ Готово к Stage 5 — Training')


✅ Training setup:
   Loss      : FocalLoss(gamma=2.0, weighted)
   Optimizer : AdamW  lr=0.0003  wd=0.001
   Scheduler : OneCycleLR  epochs=100  warmup=10%
   Patience  : 15 epochs (val macro-F1)
   Steps/epoch: 97

       STAGE 4 — COMPLETE
  Модель     : YT8MClassifier
  Параметры  : 1,107,468  (4.22 MB)
  Param/sample: 44.4x
  Loss       : FocalLoss gamma=2.0
  Optimizer  : AdamW lr=3e-4
  Scheduler  : OneCycleLR 100 epochs
✅ Готово к Stage 5 — Training
